In [1]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3
# %matplotlib inline
%matplotlib qt5
import mne
mne.viz.set_browser_backend("qt")  # or "matplotlib"
mne.set_config("MNE_BROWSER_BACKEND", "qt")  # or "matplotlib"
%gui qt

import xarray as xr # Assuming you're using this
import numpy as np   # For the example

import xarray as xr
import zarr
import panel as pn
import holoviews as hv
hv.extension('bokeh', logo=False)

import hvplot.xarray
import hvplot.pandas
# This line is crucial for displaying plots in a notebook
hvplot.extension('bokeh') # You can also use 'matplotlib' or 'plotly'

# hv.extension('bokeh')
# hv.extension('matplotlib') # or 'matplotlib'
# hv.extension('plotly') # or 'matplotlib'
from holoviews import opts
import panel as pn
pn.extension()


import IPython

# Jupyter-lab enable printing for any line on its own (instead of just the last one in the cell)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

Automatic pdb calling has been turned OFF
Using qt as 2D backend.



# Use MNE to load and analyze saved EEG and Motion recordings


In [2]:
import time
import re
from datetime import datetime, timezone

import uuid
from copy import deepcopy
from typing import Dict, List, Tuple, Optional, Callable, Union, Any
from nptyping import NDArray
from matplotlib import pyplot as plt

from pathlib import Path
import numpy as np
import pandas as pd
from numpy.typing import NDArray

import mne
from mne import set_log_level
from copy import deepcopy
import mne

from mne.io import read_raw

datasets = []
# mne.viz.set_browser_backend("Matplotlib")
mne.viz.set_browser_backend("qt")



'qt'

In [3]:
import mne
from phoofflineeeganalysis.analysis.anatomy_and_electrodes import ElectrodeHelper
from mne.channels.montage import DigMontage

electrode_pos_parent_folder: Path = Path("E:/Dropbox (Personal)/Hardware/Consumer EEG Headsets/Emotiv Epoc EEG/ElectrodeLayouts").resolve()
electrode_positions_path: Path = electrode_pos_parent_folder.joinpath('ElectrodePositions_2025-08-14', 'brainstorm_electrode_positions_PhoHAle_eeg_subjectspacemm.tsv')

active_electrode_man: ElectrodeHelper = ElectrodeHelper.init_EpocX_montage(electrode_positions_path=electrode_positions_path)
emotiv_epocX_montage: DigMontage = active_electrode_man.active_montage

head_mesh_path = Path(r"C:\Users\pho\repos\EmotivEpoc\PhoOfflineEEGAnalysis\src\phoofflineeeganalysis\resources\ElectrodeLayouts\head_bem_1922V_fill.stl").resolve()
# assert head_mesh_path.exists()
print(f'head_mesh_path: {head_mesh_path}')


# Just create montage from your electrode positions
print("Montage created successfully!")
print(f"Channel names: {emotiv_epocX_montage.ch_names}")
# Visualize the montage
ElectrodeHelper.visualize_montage(emotiv_epocX_montage)


# subjects_dir = Path(r"C:/Users/pho/Documents/MATLAB/brainstorm_database/2025-07-12_Lab_Brainstorm_Protocol01/anat/PhoHAle").resolve()

# subjects_dir = Path(r"\\wsl.localhost\Ubuntu-22.04\home\pho\freesurf_subjects\PhoHAle").resolve()
# subjects_dir = Path(r"C:/Users/pho/Documents/MATLAB/brainstorm_database/2025-07-12_Lab_Brainstorm_Protocol01/anat/PhoHAle").resolve()
# subjects_dir = Path(r"\\wsl.localhost\Ubuntu-22.04\home\pho\freesurf_subjects\PhoHAle\PhoHAle").resolve()
subjects_dir = Path(r"E:/Dropbox (Personal)/personalStore/records/Health/Ann Arbor/Pho MRI 2025-06-23/EXPORTS/From_FreeSurfer/freesurf_subjects/PhoHAle").resolve()
assert subjects_dir.exists(), f"subjects_dir: '{subjects_dir}' does not exist!"

# smooth_brain_mesh_path: Path = Path(r"E:/Dropbox (Personal)/personalStore/records/Health/Ann Arbor/Pho MRI 2025-06-23/EXPORTS/From_brain2print/brain1_smoother_mesh.obj").resolve()
# verts, faces = mne.read_surface(smooth_brain_mesh_path, file_format='obj')

# mne.write_surface('outer_skin.surf', verts, faces, overwrite=True)


Using matplotlib as 2D backend.
head_mesh_path: C:\Users\pho\repos\EmotivEpoc\PhoOfflineEEGAnalysis\src\phoofflineeeganalysis\resources\ElectrodeLayouts\head_bem_1922V_fill.stl
Montage created successfully!
Channel names: ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']


In [10]:
import numpy as np
import pyvista as pv
import pyvistaqt as pvqt
from pathlib import Path
import mne
from phoofflineeeganalysis.analysis.anatomy_and_electrodes import ElectrodeHelper

# === 1. Setup electrode montage ===
electrode_pos_parent_folder = Path("E:/Dropbox (Personal)/Hardware/Consumer EEG Headsets/Emotiv Epoc EEG/ElectrodeLayouts").resolve()
electrode_positions_path = electrode_pos_parent_folder.joinpath(
    'ElectrodePositions_2025-08-14',
    'brainstorm_electrode_positions_PhoHAle_eeg_subjectspacemm.tsv'
)
electrode_positions_path


WindowsPath('E:/Dropbox (Personal)/Hardware/Consumer EEG Headsets/Emotiv Epoc EEG/ElectrodeLayouts/ElectrodePositions_2025-08-14/brainstorm_electrode_positions_PhoHAle_eeg_subjectspacemm.tsv')

In [ ]:

active_electrode_man = ElectrodeHelper.init_EpocX_montage(electrode_positions_path=electrode_positions_path)
emotiv_epocX_montage = active_electrode_man.active_montage

# === 2. Load STL head mesh ===
head_mesh_path = Path(r"C:/Users/pho/repos/EmotivEpoc/PhoOfflineEEGAnalysis/src/phoofflineeeganalysis/resources/ElectrodeLayouts/head_bem_1922V_fill.stl").resolve()

head_mesh = pv.read(str(head_mesh_path))
print(f"Loaded STL mesh with {head_mesh.n_points} vertices and {head_mesh.n_cells} faces.")

# === 2b. Fix STL scaling and centering ===
# Convert from mm → m
if np.max(np.abs(head_mesh.points)) > 0.1:  # heuristic threshold
# if np.max(np.abs(mesh.bounds)) > 1:  # heuristic: STL likely in mm
    head_mesh.points *= 1e-3

# Center mesh at origin
head_mesh_center = head_mesh.center
head_mesh.translate(-np.array(head_mesh_center), inplace=True)
print(f"Mesh center shifted by {-np.array(head_mesh_center)} (now centered at {head_mesh.center})")


# === 3. Convert montage into XYZ points ===
ch_pos = emotiv_epocX_montage.get_positions()['ch_pos']
electrode_points = np.array(list(ch_pos.values()))

electrode_points[:, 1] = electrode_points[:, 1] - 0.04
electrode_points[:, 2] = electrode_points[:, 2] + 0.052

ch_names = list(ch_pos.keys())

ch_pos_df: pd.DataFrame = pd.DataFrame(ch_pos).T
ch_pos_df.columns = ['x', 'y', 'z']
ch_pos_df

# # Convert to meters if in mm
# if np.max(np.abs(points)) > 0.1:  # heuristic threshold
#     points *= 1e-3

# === 4. Compute normals and find nearest surface points ===
# Compute normals for the head mesh
head_mesh = head_mesh.compute_normals(point_normals=True, cell_normals=False, consistent_normals=True)

# For each electrode point, find the nearest point on the mesh surface and get its normal
nearest_points = np.zeros_like(electrode_points)
normals = np.zeros_like(electrode_points)

for i, electrode_point in enumerate(electrode_points):
    # Find closest vertex on mesh
    closest_vertex_idx = head_mesh.find_closest_point(electrode_point)
    closest_point = head_mesh.points[closest_vertex_idx]
    
    # Get normal at the closest vertex
    normal = head_mesh.point_normals[closest_vertex_idx]
    
    nearest_points[i] = closest_point
    normals[i] = normal

# === 5. Visualize with PyVista ===
# plotter = pv.Plotter()
plotter = pvqt.BackgroundPlotter()
a_mesh = plotter.add_mesh(head_mesh, 
            # color='lightgray', opacity=0.3,
            color='lightgray', opacity=0.9,

)

# Create oriented glyphs (cones) at nearest surface points, oriented along normals
# Create a PolyData object with points and normals
electrode_glyphs = pv.PolyData(nearest_points)
electrode_glyphs['normals'] = normals

# Create cone glyphs oriented along normals
cone = pv.Cone(radius=0.005, height=0.01, resolution=8)  # Larger, more visible cones
a_points = plotter.add_mesh(electrode_glyphs.glyph(orient='normals', scale=False, factor=1.0, geom=cone),  color='red', show_edges=False)

# Add text labels for each electrode (at the nearest surface points) with transparent background
a_labels = plotter.add_point_labels(nearest_points, ch_names, font_size=10, text_color='black', point_color='red', always_visible=True, shape_opacity=0.0)

# Optional visual tweaks
# plotter.show_grid(visible=False)
plotter.show_axes()
plotter.set_background("white")
title="EEG electrodes + head mesh"

plotter.show()


In [ ]:
ch_pos_df

In [ ]:

brain_kwargs = dict(alpha=0.5, subjects_dir=subjects_dir,
    hemi="lh", surf="pial", size=(800, 600),
)
Brain = mne.viz.get_brain_class()
brain = Brain("PhoHAle", **brain_kwargs)
brain.add_annotation("aparc.a2009s", borders=False)

In [ ]:

subjects_dir = mne.datasets.sample.data_path()
brain_kwargs = dict(alpha=0.5, subjects_dir=subjects_dir)

brain = mne.viz.Brain("sample", subjects_dir=subjects_dir, **brain_kwargs)
brain.add_head(alpha=0.5)


In [ ]:

from mne_lsl.player import PlayerLSL as Player
from mne_lsl.stream import StreamLSL as Stream

from phoofflineeeganalysis.analysis.MNE_helpers import MNEHelpers
from phoofflineeeganalysis.analysis.historical_data import HistoricalData
from phoofflineeeganalysis.analysis.motion_data import MotionData
from phoofflineeeganalysis.analysis.EEG_data import EEGComputations, EEGData
from phoofflineeeganalysis.analysis.anatomy_and_electrodes import ElectrodeHelper
# from ..EegProcessing import bandpower
# from phoofflineeeganalysis.EegProcessing import analyze_eeg_trends
from phoofflineeeganalysis.EegVisualization import VisHelpers
from phoofflineeeganalysis.analysis.SavedSessionsProcessor import SavedSessionsProcessor, SessionModality, DataModalityType

set_log_level("WARNING")


# db_root_path = Path('/content/drive/MyDrive/Databases').resolve()
db_root_path = Path(r'E:/Dropbox (Personal)/Databases').resolve()
assert db_root_path.exists(), f"'{db_root_path.as_posix()}' does not exist!"

# eeg_recordings_file_path: Path = Path(r'E:/Dropbox (Personal)/Databases/UnparsedData/EmotivEpocX_EEGRecordings/fif').resolve()
# headset_motion_recordings_file_path: Path = Path(r'E:/Dropbox (Personal)/Databases/UnparsedData/EmotivEpocX_EEGRecordings/MOTION_RECORDINGS/fif').resolve()

# assert eeg_recordings_file_path.exists()
# assert headset_motion_recordings_file_path.exists()

eeg_recordings_file_path: Path = db_root_path.joinpath('UnparsedData/EmotivEpocX_EEGRecordings/fif').resolve()
flutter_eeg_recordings_file_path: Path = db_root_path.joinpath('UnparsedData/EmotivEEG_FlutterRecordings').resolve()
flutter_motion_recordings_file_path: Path = db_root_path.joinpath('UnparsedData/EmotivEEG_FlutterRecordings/MOTION_RECORDINGS').resolve()
flutter_GENERIC_recordings_file_path: Path = db_root_path.joinpath('UnparsedData/EmotivEEG_FlutterRecordings/GENERIC_RECORDINGS').resolve()

headset_motion_recordings_file_path: Path = db_root_path.joinpath('UnparsedData/EmotivEpocX_EEGRecordings/MOTION_RECORDINGS/fif').resolve()
WhisperVideoTranscripts_LSL_Converted = db_root_path.joinpath('UnparsedData/WhisperVideoTranscripts_LSL_Converted').resolve()
pho_log_to_LSL_recordings_path: Path = db_root_path.joinpath('UnparsedData/PhoLogToLabStreamingLayer_logs').resolve()
## These contain little LSL .fif files with names like: '20250808_062814_log.fif',

eeg_analyzed_parent_export_path = db_root_path.joinpath('AnalysisData/MNE_preprocessed').resolve()
pickled_data_path = db_root_path.joinpath('AnalysisData/MNE_preprocessed/PICKLED_COLLECTION').resolve()
assert pickled_data_path.exists()


# n_most_recent_sessions_to_preprocess: int = None # None means all sessions
# n_most_recent_sessions_to_preprocess: int = 35
# n_most_recent_sessions_to_preprocess: int = 5
n_most_recent_sessions_to_preprocess = None
